<a href="https://colab.research.google.com/github/ryoteguh/PROJECT-TOURISM-C/blob/main/Project%20sentiment%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.makedirs('templates', exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>Sentiment Analysis Review</title>
    <style>
        body { font-family: sans-serif; background: #f4f4f9; padding: 20px; }
        .container { max-width: 800px; margin: auto; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
        h2 { color: #333; }
        input, button { padding: 10px; margin: 5px 0; border-radius: 5px; border: 1px solid #ddd; }
        button { background: #007bff; color: white; cursor: pointer; border: none; }
        table { width: 100%; border-collapse: collapse; margin-top: 20px; }
        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }
        th { background: #f8f9fa; }
        .pos { color: green; font-weight: bold; }
        .neu { color: orange; font-weight: bold; }
        .neg { color: red; font-weight: bold; }
    </style>
</head>
<body>
    <div class="container">
        <h2>📊 Analisis Sentimen Play Store</h2>
        <form action="/analyze" method="post">
            <input type="text" name="app_id" value="com.tiket.gits" style="width: 70%;">
            <input type="number" name="count" value="5" style="width: 20%;">
            <button type="submit">Analisis Sekarang</button>
        </form>

        {% if data %}
        <table>
            <tr>
                <th>User</th>
                <th>Komentar</th>
                <th>Sentimen</th>
            </tr>
            {% for item in data %}
            <tr>
                <td>{{ item.user }}</td>
                <td>{{ item.text }}</td>
                <td class="{{ item.class }}">{{ item.sentiment }}</td>
            </tr>
            {% endfor %}
        </table>
        {% endif %}
    </div>
</body>
</html>

Overwriting templates/index.html


In [ ]:
%%writefile app.py
from flask import Flask, render_template, request
from transformers import pipeline
from google_play_scraper import Sort, reviews
import pandas as pd

app = Flask(__name__)

# Load Model
baksoayam = pipeline("sentiment-analysis", model="mdhugol/indonesia-bert-sentiment-classification")
kamus = {'LABEL_0': 'Positive', 'LABEL_1': 'Neutral', 'LABEL_2': 'Negative'}
kelas = {'LABEL_0': 'pos', 'LABEL_1': 'neu', 'LABEL_2': 'neg'}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/analyze', methods=['POST'])
def analyze():
    app_id = request.form.get('app_id')
    count = int(request.form.get('count'))

    # Scrapping
    result, _ = reviews(app_id, lang='id', country='id', sort=Sort.NEWEST, count=count)

    formatted_data = []
    for r in result:
        # Prediksi AI
        res_ai = baksoayam(r['content'])[0]
        formatted_data.append({
            'user': r['userName'],
            'text': r['content'],
            'sentiment': kamus[res_ai['label']],
            'class': kelas[res_ai['label']]
        })

    return render_template('index.html', data=formatted_data)

if __name__ == '__main__':
    app.run(port=5000)

Overwriting app.py


In [ ]:
!pip install flask transformers google-play-scraper accelerate
!npm install -g localtunnel

# Jalankan Flask di background dan buka tunnel ke port 5000
import urllib
print("Password IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!python app.py & npx localtunnel --port 5000

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 1s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏Password IP: 34.125.27.140
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://real-nights-wash.loca.lt
Loading weights: 100% 201/201 [00:00<00:00, 984.40it/s, Materializing param=classifier.weight] 
BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
 * Serving Flask app 'app'
 * Debug mode: off
Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
